In [1]:
# Import the main libraries

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
from pyproj import CRS
import matplotlib.pyplot as plt
import folium
from folium.raster_layers import ImageOverlay
import rasterio
from branca.element import Template, MacroElement



In [2]:
# Importing a file and converting a gdf

### Function which returned a geodataframe from a geopackage o GeoJSON file and print the head, the inputs are the path of the geopackage or geojson file
### and the layer inside the geopackage


def create_gdf(path, layer=None):
    '''
    The function reads a geopackage or geojson and convert it into a GeoDataFrame.
    
    parameters: 
        path (str): Path to the geopackage file.
        layer (str): If the geopackage has different layers, specify the layer to open, otherwise layer = None.
    
    Print:
        print(gdf_head): It prints the first five rows for the geodataframe as a reference.
        
    Returns:
        gdf (GeoDataFrame): It contains the GeoDataFrame with all the information in the geopackage or geojson file.
        gdf_head (GeoDataFrame): It contains the first rows in the gdf, they are ready to print. 
    
    Raises: 
        FileNotFoundError: It will raise, if the path does not exist or it is wrong.
        ValueError: It will raise, if the file cannot be read by geopandas because the name of the layers.
    '''
    
    if not os.path.exists(path):
            raise FileNotFoundError(f'The file does not exist or the path is wrong:{path}')
    
    if path.lower().endswith(".gpkg"):
        layers = fiona.listlayers(path)
        
        if layer is None:
             raise ValueError(f'geopackage contains multiple layers: {layers}, specify the layer to read')
        
        if layer not in layers:
            raise ValueError(f'layer {layer} not found. Available leyers:{layers}')
        
    try:    
        gdf = gpd.read_file(path, layer=layer)   # This part was fixed update the code 
                
    except Exception as e:
        # Raise a error if for any reason geopandas cannot read the geopackage
        raise ValueError(f'Error reading the file {path}: {e}')
    
    
    return gdf

# Administrative boundary Derna
Adm_bound = create_gdf(r'C:\MGEO\Y1\Q2\Scientific Prog Geospatial Sciences\Programming exercises\Week_10\Assignment_2_DMR\Sci_Prog_Group_Assignment2\datasets\inputs\Adm_Bound.gpkg', layer='ADM_ADM_1')

# Flood extension 
flood_ext = create_gdf(r'C:\MGEO\Y1\Q2\Scientific Prog Geospatial Sciences\Programming exercises\Week_10\Assignment_2_DMR\Sci_Prog_Group_Assignment2\datasets\inputs\PHR_20230913_FloodExtent_Derna.shp')

# Roads Derna
Roads_Derna = create_gdf(r'C:\MGEO\Y1\Q2\Scientific Prog Geospatial Sciences\Programming exercises\Week_10\Assignment_2_DMR\Sci_Prog_Group_Assignment2\outputs\impacted_roads.gpkg', layer ='impacted_roads')

# Buildings Derna
Buildings_Derna = create_gdf(r"C:\MGEO\Y1\Q2\Scientific Prog Geospatial Sciences\Programming exercises\Week_10\Assignment_2_DMR\Sci_Prog_Group_Assignment2\outputs\impacted_buildings_with_water_depth.gpkg", layer= 'impacted_buildings_with_water_depth')


In [3]:
### Function to filter the GeoDataFrame administrative boundary only for the distric of Derna

def filter_district(gdf, distric):
    '''
    The function reads a GeoDataFrame and district name choosen and filter the data only for the data inside the distric name.
    
    parameters: 
        gdf (geodataframe): GeoDataFrame to apply the filter.
        district (str): Name of the district for the analysis. 
    
    Print:
        print(the number of records): It prints the number of the records after applying the function.
                
    Returns:
        gdf (GeoDataFrame): It contains the GeoDataFrame with the data after applying the function.
            
    Raises: 
        ValueError: It will raise, if the district name is not in the GeoDataFrame.
        TypeError: It will raise, if the input is not a GeoDataFrame.
    '''
    
    if  not (gdf['NAME_1'].str.lower() == distric.lower()).any():
        raise ValueError(f'The province: {distric}, is not in the geodataframe')
     
    elif not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError("Input must be a valid GeoDataFrame")
    
    else:
        gdf = (gdf[gdf['NAME_1'].str.lower() == distric.lower()])  # Filter function
        print(f'the number of records: {len(gdf)}')
        return gdf

# Filtering the District to Darnah
Adm_bound_fil = filter_district(Adm_bound, 'Darnah')

the number of records: 1


In [5]:
# Function to validate the CRS project (4326) in a GeoDataFrame and if the CRS is not that, it will be reprojected to this CRS project. (function assignment 1)
# This step is neccesary for plotting using folium

def check_crs_4326(gdf):
    '''
    The function reads a GeoDataFrame and considerer a predefine CRS based on EPSG for the project then it reads the current crs and if 
    this is different from the defined parameter, it reproject the geodataframe.
    
    parameters: 
        gdf (geodataframe): GeoDataFrame to validate and set the coordinate system
    
    Print:
        print(crs_initial): It prints the original CRS for the geodataframe.
        print(crs_final): It prints the final CRS for the geodataframe after verification.
        
    Returns:
        gdf (GeoDataFrame): It contains the GeoDataFrame with the CRS setup for the project after verification.
            
    Raises: 
        ValueError: It will raise, if the GeoDataFrame does not have a CRS defined.
    '''
    
    if gdf.crs is None:
        raise ValueError("NO CRS defined.")
    crs_initial = gdf.crs
    
    # Set the CRS target for the project
    target_crs = CRS.from_epsg(4326)
    
    # Set the CRS for the project in case it is different to the target.
    if gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)    # I suggest this improvement
        
    print(f'Initial CRS: {crs_initial}') 
    print(f'Final CRS: {gdf.crs}') 
    return gdf


# GeoDataFrame Administrative boundary
Adm_bound_Derna_4326= check_crs_4326(Adm_bound_fil)

# Flood extension 
flood_ext_4326 = check_crs_4326(flood_ext)

# Roads Derna
Roads_Derna_4326 = check_crs_4326(Roads_Derna)

# Buildings Derna
Buildings_Derna_4326 = check_crs_4326(Buildings_Derna)

Initial CRS: EPSG:4326
Final CRS: EPSG:4326
Initial CRS: EPSG:4326
Final CRS: EPSG:4326
Initial CRS: EPSG:3177
Final CRS: EPSG:4326
Initial CRS: EPSG:32634
Final CRS: EPSG:4326


In [6]:

#### Function which takes four geodataframes in WGS 84 and plotted them to produce an interactive flood exposure map.
#### The function also overlay a basemap tiles

# Plotting a map exposure for Derna and in the final part print summaries (pending the stadistics).

#def plot_exposure(gdf, titles, base_layer=None, overlay_layers=None):
def plot_exposure(gdf, base_layer=None, overlay_layers1=None, overlay_layers2=None):
    '''
    The function reads four GeoDataFrames related to vector files and xxx files and plotted an interactive map with a specific legend and title set 
    Also, the function use a baselayer as default with a satellite imagery for visualization
    
    parameters: 
        gdf (geodataframe): GeoDataFrame to plot a map.
        base_layer (geodataframe): Additional GeoDataFrame to plot on the map.
        overlay_layers1 (geodataframe): Additional 1 GeoDataFrame to plot on the map.
        overlay_layers2 (geodataframe): Additional 2 GeoDataFrame to plot on the map.
                            
    Print: 
        This function does not print information.
                
    Returns:
        m (plot): It displays the interactive map with the different layers.
                
    Raises: 
        TypeError: It will raise, if one of the inputs are not a GeoDataFrame or titles is not strings.
    '''
        
    if not isinstance(gdf, gpd.GeoDataFrame):
            raise TypeError("Input must be a valid GeoDataFrame")
    
    if base_layer is None or overlay_layers1 is None or overlay_layers2 is None:
        pass
    else:
        if not isinstance(base_layer, gpd.GeoDataFrame) or not isinstance(overlay_layers1, gpd.GeoDataFrame) or not isinstance(overlay_layers2, gpd.GeoDataFrame):
            raise TypeError("Inputs must be a valid GeoDataFrame")
        
                
    # Copy the geodataframes to plot and filter some attributes
    gdf_plot = gdf.copy()
    base_layer_plot = base_layer.copy()
    base_layer_plot = base_layer_plot.drop(columns=['Sensor_Dat'])
    overlay_layers1_plt = overlay_layers1.copy()
    overlay_layers2_plt = overlay_layers2.copy()
    overlay_layers2_plt_flood = overlay_layers2_plt[overlay_layers2_plt['status'] == 'Flooded']
    overlay_layers2_plt_dry = overlay_layers2_plt[overlay_layers2_plt['status'] == 'Dry']
      
   # Define centroid for visualization
    centroid = base_layer.geometry.centroid.iloc[0]
    cx, cy = centroid.x, centroid.y
    deltax= 2000/111111 # length define in meter to plotting and divide by 111,111 to convert to degrees
    deltay= 2500/111111 # length define in meter to plotting and divide by 111,111 to convert to degrees
    m = folium.Map(location=[cy+deltay, cx+deltax], zoom_start=14)
    
    # Add some folium Tilelayer, in this case from ESRI
    folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri',
    name='Esri Satellite',
    overlay=False,
    control=True
    ).add_to(m)   # arcgis imagery
    
    # Define the layers to overlay in the main map
    layer1 = folium.FeatureGroup(name="Administrative boundary")
    folium.GeoJson(gdf_plot, style_function=lambda f:{'fillColor': 'transparent', 'color': 'black', 'weight': 2, 'fillOpacity': 0.5}).add_to(layer1)
    
    #folium.GeoJson(base_layer).add_to(m)
    layer2 = folium.FeatureGroup(name="Flood extent")
    folium.GeoJson(base_layer_plot).add_to(layer2)
    #gdf_plot = gdf.copy().reset_index(drop=True)
    
    layer3 = folium.FeatureGroup(name="Road affected")
    folium.GeoJson(overlay_layers1_plt, style_function=lambda f:{'color': 'red', 'weight': 1}).add_to(layer3)
    
    layer4 = folium.FeatureGroup(name="Buildings flooded")
    folium.GeoJson(overlay_layers2_plt_flood, marker=folium.CircleMarker(radius=0.4, color='blue', fill=True, fill_opacity=0.7), tooltip=folium.GeoJsonTooltip(fields=['water_depth'], aliases=['Water depth (m):'], localize=True, sticky=True)).add_to(layer4)
    
    layer5 = folium.FeatureGroup(name="Buildings dry")
    folium.GeoJson(overlay_layers2_plt_dry, marker=folium.CircleMarker(radius=0.4, color='gray', fill=True, fill_opacity=0.9), tooltip=folium.GeoJsonTooltip(fields=['water_depth'], aliases=['Water depth (m):'], localize=True, sticky=True)).add_to(layer5)
    
      
    # Add the layers to the main map
    layer1.add_to(m)
    layer2.add_to(m)
    layer3.add_to(m)
    layer4.add_to(m)
    layer5.add_to(m)
    
    # HTML template for the title
    title_html = """
    {% macro html(this, kwargs) %}
    <div style="
    position: fixed; 
    top: 10px; left: 50px; width: 90%; 
    z-index:9999; 
    font-size:24px; 
    font-weight:bold;
    background-color: rgba(255, 255, 255, 0.7);
    padding: 5px;
    border-radius:5px;
    text-align:center;
    ">
    Project: Flood exposure map Derna, Libya
    </div>
    {% endmacro %}
    """
    # Add the title to the map
    title = MacroElement()
    title._template = Template(title_html)
    m.get_root().add_child(title)
    
    folium.LayerControl().add_to(m)
   
          
    return m
    



In [7]:
# Test the function 

plot_exposure(Adm_bound_Derna_4326,  base_layer=flood_ext_4326, overlay_layers1=Roads_Derna_4326, overlay_layers2=Buildings_Derna_4326)

C:\Users\david\AppData\Local\Temp\ipykernel_18960\3513011864.py:48: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = base_layer.geometry.centroid.iloc[0]
